# v004 · Train từng model (2 hàng song song), test ngay trên test cả 3 part, báo cáo

Mỗi phiên (`GROUP` = `P1`…`P5`, `settings.TRAIN_SESSIONS`) là **một notebook Kaggle T4 ×2** tên `vn-av-df-train-<GROUP>`: hai hàng song song, mỗi hàng một GPU, các bước trong hàng chạy nối tiếp. **Model nào train xong thì test ngay** trên tập test của cả 3 part (in điểm, ghi `metrics.json`, run W&B `evaluate`), không chờ hàng kia. Mỗi bước là một model (`settings.TRAIN_GROUPS`); bỏ bước nào thì thêm vào `DROP`. Chạy một bước lẻ: `GROUP = "<bước>"` (vd. `"M2"`). Giờ ước, model nào bỏ được: `RESEARCH_WORKLOG/v004_model.md` §8.2.

| GROUP | Accelerator | GPU0 | GPU1 |
|---|---|---|---|
| `labels` | None (CPU) | nhãn mismatch 0 / 1 / mask + kiểm lối tắt độ trễ + kiểm ngữ điệu–đầu → `sync_labels.json` (gửi xem trước khi train) | |
| `P1` | T4 ×2 | M3 (+ benchmark mọi model) | M3 −prosody |
| `P2` (song song P1 được) | T4 ×2 | BL-M/sup | BL-A2 → BL-V → M2 → BL-M + SpeechForensics |
| `P3` | T4 ×2 | M1 (dừng ở `MAX_HOURS`) | M3 −Φ → M3 −abs |
| `P4` (`PREVIOUS_RUN` = P3) | T4 ×2 | M1 chạy tiếp | M3-ref |
| `P5` (nếu giữ BL-A; `PREVIOUS_RUN` = P4) | T4 ×2 | BL-A | M3-ref / M1 chạy tiếp nếu còn dở |
| `report` | None (CPU) | gộp Output mọi phiên (`PREVIOUS_RUN`): ghép muộn BL-Ens / BL-Ens-AV, cổng V1-V11, `report.md` (không chấm lại) | |

Cần: 3 dataset part (có `sync_syncnet.json`, chạy `training/01a_syncnet.py` ở local) + Output 9 phiên prepare. Mỗi phiên ghi run W&B `setup-train-<GROUP>-<giờ>` (cách cài phiên); mỗi run train ghi `architecture.json` + `setup_run`; mỗi lần test ghi `test-log.json` (số lần model dùng test). Model dừng ở `MAX_HOURS` chưa xong thì chưa test; phiên sau train tiếp rồi test.

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json, shutil
SESSION_START = time.time()
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa thành repository của bạn.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
WORKERS = Path("/kaggle/temp/vn-av-df-workers")  # Môi trường worker: ngoài Output 20 GB.
# Log cài đặt của phiên (git, pip, worker): ghi đủ ra file, cell "Log setup" đưa lên W&B + Output.
LOGS = Path("/kaggle/temp/vn-av-df-logs")
LOGS.mkdir(parents=True, exist_ok=True)
TIMINGS = {}
def logged(name, command, **kw):
    tick = time.time()
    path = LOGS / f"{name}.log"
    with open(path, "a", encoding="utf-8") as log:
        log.write("$ " + " ".join(map(str, command)) + "\n")
        log.flush()
        code = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT, **kw).returncode
    TIMINGS[name] = round(TIMINGS.get(name, 0) + time.time() - tick, 1)
    if code:
        print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"{name} lỗi (mã {code}); log đầy đủ: {path}")
if not ROOT.exists():
    logged("git-clone", ["git", "clone", REPO_URL, str(ROOT)])
# Weight và repo upstream (vài GB) ở /kaggle/temp: Output (giới hạn 20 GB) không bao giờ chứa
# chúng, kể cả khi phiên lỗi giữa chừng (bước dọn Output cuối không chạy).
for name in ("weights", "external"):
    (Path("/kaggle/temp/vn-av-df-heavy") / name).mkdir(parents=True, exist_ok=True)
    if not (ROOT / name).exists():
        (ROOT / name).symlink_to(Path("/kaggle/temp/vn-av-df-heavy") / name, target_is_directory=True)
sys.path.insert(0, str(ROOT))
logged("pip-install", [sys.executable, "-m", "pip", "install", "-e", str(ROOT) + "[data,training]"])
import settings
settings.bootstrap()

In [ ]:
# W&B: log quá trình (setup từng phiên, trích feature, nhãn, train, test, báo cáo) làm bằng chứng.
# Project PRIVATE. Thêm secret WANDB_API_KEY: Add-ons → Secrets. Không đẩy ảnh / video người thật.
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-v004"  # v003: "vn-av-df-v003" (giữ nguyên, không ghi chồng)
WANDB_ENTITY = "vn-av-df-capstone"  # team trong org vn-av-df-capstone-org; key tạo trong org đó
if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_DIR"] = "/kaggle/temp"  # file tạm W&B ở ngoài Output
    os.makedirs(os.environ["WANDB_DIR"], exist_ok=True)
WANDB = {"project": WANDB_PROJECT, "entity": WANDB_ENTITY} if USE_WANDB else None

In [ ]:
GROUP = "labels"      # "labels" → "P1" (∥ "P2") → "P3" → "P4" → ("P5") → "report"; hoặc một bước lẻ, vd. "M2"
DROP = []             # bước bỏ ở phiên này, vd. ["M3-abs"] (settings.TRAIN_GROUPS)
DATASET_PARTS = [Path(f"/kaggle/input/datasets/linhxm/{s}") for s in (
    "vn-av-df-data-part1-trim04-fake", "vn-av-df-data-part2-fake", "vn-av-df-data-part3-fake")]
# Output của 9 phiên prepare (3 part × landmarks / cpu / gpu), tên notebook vn-av-df-prepare-p<N>-<phiên>.
PREPARE = "/kaggle/input/notebooks/linhxm/vn-av-df-prepare-p{}-{}/vn-av-df-forensics/cache/vn-av-df-data"
CACHES = [Path(PREPARE.format(p, s)) for p in (1, 2, 3) for s in ("landmarks", "cpu", "gpu")]
RUNS = "/kaggle/input/notebooks/linhxm/vn-av-df-train-{}/vn-av-df-forensics/runs/" + settings.RUN_NAME
PREVIOUS_RUN = []     # P4: [Path(RUNS.format("P3"))]; P5: [Path(RUNS.format("P4"))];
                      # report: [Path(RUNS.format(g)) for g in ("P1", "P2", "P3", "P4")] (+ "P5" nếu chạy)
SEEDS = [42]          # lượt đầu một seed; chốt thiết kế rồi chạy [42, 43, 44]
MAX_HOURS = 11        # tính từ cell này; dừng sạch (lưu last.pt) trước giới hạn 12 giờ của Kaggle
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # trước lần dùng GPU đầu
import torch
from vn_av_df.dataset import part_selection
for c in CACHES:
    assert c.is_dir(), f"Sai đường dẫn: {c}"
settings.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if GROUP not in ("labels", "report", *settings.TRAIN_SESSIONS, *settings.TRAIN_GROUPS):
    raise ValueError(f"GROUP không có: {GROUP}")
if GROUP not in ("labels", "report") and not torch.cuda.is_available():
    raise RuntimeError(f"GROUP {GROUP} cần Accelerator GPU")
cfg = settings.config()
cfg["deadline"] = time.time() + MAX_HOURS * 3600
# BL-A trọng số lớp: "balanced" = 1 − tỉ lệ lớp trong train, cùng nguyên tắc với gốc; "paper" =
# đúng 0,1 / 0,9 (cân bằng cho ASVspoof ~90 % giả). Đổi thì BL-A train lại từ đầu.
cfg["recipe_overrides"] = {"ssl_aasist": {"class_weight": "balanced"}}
root, parts = part_selection(DATASET_PARTS)
cfg["dataset"], cfg["dataset_parts"] = str(root), parts
cfg["cache"] = [str(p) for p in CACHES]
# Đội hình đầy đủ (settings.TEST_MODELS): để in model nào chưa xong; mỗi bước chỉ train / test model của nó.
cfg["models"], cfg["seeds"] = list(settings.TEST_MODELS), list(SEEDS)
cfg["wandb"] = WANDB
SESSION_NAME = f"train-{GROUP}-{time.strftime('%Y%m%d-%H%M')}"
cfg["session"], cfg["train_group"] = SESSION_NAME, GROUP
from vn_av_df.training.train import merge_runs, missing_runs
from vn_av_df.setup_log import copy_tree
previous = [PREVIOUS_RUN] if isinstance(PREVIOUS_RUN, (str, Path)) else list(PREVIOUS_RUN or [])
for p in previous:
    assert Path(p).is_dir(), f"Sai đường dẫn: {p}"
if previous:
    print("Gộp từ phiên trước:", merge_runs(cfg["runs"], previous))
# Log setup của 9 phiên prepare đi cùng kết quả train (runs/<RUN>/setup/prepare-…).
print("Log setup prepare:", sum((copy_tree(c / "setup", Path(cfg["runs"]) / "setup") for c in CACHES), []))
if GROUP in ("labels", "report"):
    STEPS = []
elif GROUP in settings.TRAIN_SESSIONS:
    STEPS = [[s for s in lane if s not in DROP] for lane in settings.TRAIN_SESSIONS[GROUP]]
else:
    STEPS = [[GROUP]]
# Mỗi hàng một GPU: model của các bước trong hàng, chạy nối tiếp; xong model nào test ngay model đó.
LANES = [[m for s in lane for group in settings.TRAIN_GROUPS[s]["lanes"] for m in group] for lane in STEPS]
LANES = [lane for lane in LANES if lane]
MODELS = [m for lane in LANES for m in lane]  # train ở phiên này
RUN_BENCHMARK = any(settings.TRAIN_GROUPS[s].get("benchmark") for lane in STEPS for s in lane)
cfg["test_after_train"] = True
print("Phiên", GROUP, "| bước:", STEPS, "| model theo GPU:", LANES, "| chưa train xong:", missing_runs(cfg))

In [ ]:
from vn_av_df.cli import model_assets
tick = time.time()
ASSETS = model_assets(cfg) if MODELS or RUN_BENCHMARK else {}  # XLS-R gốc (BL-A) + checkpoint AVH-Align (BL-M, M2, M3)
TIMINGS["assets"] = round(time.time() - tick, 1)
print(ASSETS)

## Log setup

In [ ]:
SETUP_FOLDER = Path(cfg["runs"]) / "setup"
SETUP_INFO = {"group": GROUP, "steps": STEPS, "drop": DROP,
              "titles": {s: settings.TRAIN_GROUPS[s]["title"] for lane in STEPS for s in lane},
              "lanes": LANES, "benchmark": RUN_BENCHMARK, "test_models": cfg["models"], "seeds": SEEDS,
              "previous_run": [str(p) for p in previous], "datasets": [str(p) for p in DATASET_PARTS],
              "caches": [str(p) for p in CACHES], "assets": ASSETS, "max_hours": MAX_HOURS,
              "recipe_overrides": cfg["recipe_overrides"]}

# Log setup của phiên lên W&B (run job_type "setup") + Output: commit, phần cứng, gói pip của kernel
# và từng worker, log cài (pip, worker), dữ liệu / cache / tài sản gắn vào. Người đọc W&B biết mỗi
# phiên (mỗi lượt train) được cài thế nào.
from vn_av_df.setup_log import log_setup
SETUP = log_setup(cfg, SESSION_NAME, SETUP_FOLDER, logs=sorted(LOGS.glob("*.log")), env_root=WORKERS,
                  info=SETUP_INFO, timings=TIMINGS)
cfg["setup_run"] = {k: SETUP[k] for k in ("session", "wandb_id")}

In [ ]:
if GROUP == "labels":
    from vn_av_df.training.sync_labels import report
    labels = report(cfg)
    print(json.dumps({k: v for k, v in labels.items() if k != "samples"}, indent=1, ensure_ascii=False)[:20000])
    print("→ gửi runs/" + settings.RUN_NAME + "/sync_labels.json để xem trước khi train")
if RUN_BENCHMARK:
    from vn_av_df.training.benchmark import benchmark
    print(json.dumps(benchmark(cfg, cfg["models"], steps=10), indent=2))
if MODELS and torch.cuda.device_count() >= len(LANES) > 1:
    from vn_av_df.training.lanes import run_lanes
    run_lanes(cfg, LANES)
elif MODELS:
    from vn_av_df.training.train import train_all
    print(json.dumps(train_all({**cfg, "models": MODELS}), indent=2, ensure_ascii=False))

In [ ]:
# Mỗi model đã được test ngay khi train xong (trong hàng của nó); ở đây gộp kết quả (đọc lại, không chấm thêm).
from vn_av_df.training.evaluate import evaluate_all
from vn_av_df.training.train import run_state
runs = Path(cfg["runs"])
done = [m for m in MODELS if all(run_state(runs / f"{m}_seed{s}")[0] for s in SEEDS)]
if done:
    results = evaluate_all(cfg, models=done)
    print(json.dumps({k: v for k, v in results.items() if k.rsplit("_seed", 1)[0] in done}, indent=2))
elif MODELS:
    print("Chưa train xong, chưa test:", MODELS)
if GROUP == "report":
    from vn_av_df.training.report import build_report
    print("Chưa train xong (không có trong báo cáo):", missing_runs(cfg))
    print(json.dumps(evaluate_all(cfg), indent=2))  # đọc lại kết quả từng model, tính ghép muộn
    report = build_report(cfg)
    print(Path(report["report"]).read_text(encoding="utf-8"))
from vn_av_data.common.runtime import prune_output
print("Đã xoá khỏi Output:", len(prune_output(ROOT, [str(Path(cfg["runs"]).relative_to(ROOT))])), "mục")